In [4]:
# Biblitecas 
import os
import arcpy
import numpy as np
import pandas as pd
import mapclassify as mc
import math
from arcpy.sa import *

# Ativar extensão Spatial Analyst
arcpy.CheckOutExtension("Spatial")

# Caminhos gdb e Dataset 
gdb = r"F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Multi_Criterio_SES.gdb"
gdb_rasters = r"F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Rasters_IDW.gdb"
dataset = r"Multi_Criterio_SES" # Alterar no Bloco do código específico quando necessário! 

# Caminhos mapa e projeto
aprx = arcpy.mp.ArcGISProject("CURRENT")
mapa = aprx.activeMap

GRID

In [2]:
# Criando o Grid

In [3]:
# Entrada
area_estudo = "Area_PAS"

# Caminho de saída
fishnet_saida = f"{gdb}\\{dataset}\\Grid_100m"

# Obter extensão da área de estudo
desc = arcpy.Describe(area_estudo)
extent = desc.extent

origin_coord = f"{extent.XMin} {extent.YMin}"
y_axis_coord = f"{extent.XMin} {extent.YMin + 100}"
corner_coord = f"{extent.XMax} {extent.YMax}"

# Criar fishnet
arcpy.management.CreateFishnet(
    out_feature_class=fishnet_saida,
    origin_coord=origin_coord,
    y_axis_coord=y_axis_coord,
    cell_width=100,
    cell_height=100,
    number_rows="",
    number_columns="",
    corner_coord=corner_coord,
    labels="NO_LABELS",
    template=area_estudo,
    geometry_type="POLYGON"
)

print("✅ Grid 100x100m criado com sucesso.")

✅ Grid 100x100m criado com sucesso.


In [4]:
# Filtrar Camadas de Rede

In [10]:
# Conduto por Gravidade / Trechos  →  Saída: "Trecho_Redes_SES_BDG"

input_trecho = "Conduto por Gravidade BDG"

sql_trecho = """
lifecyclestatus = 8 AND ASSETGROUP IN (1) AND ASSETTYPE IN (1, 2, 3, 4, 5 , 6)
"""

layer_trecho = "Redes_SES_BDG"
arcpy.MakeFeatureLayer_management(input_trecho, layer_trecho, sql_trecho)

print("✅ Camada temporária criada: Trecho_Redes_SES_BDG")

✅ Camada temporária criada: Trecho_Redes_SES_BDG


In [6]:
# Selecionando Grids que Intersectam Redes e Ligações

In [7]:
# Entradas
input_fc = "Grid_100m"
redes = "Redes_SES_BDG"
ligacoes = "Ligacoes_BDG"

# Saída
output_fc = os.path.join(gdb, dataset, 'Grid_Selecionado')

# Criar layer (necessário para seleção)
arcpy.management.MakeFeatureLayer(input_fc, "layer_temp")

# Seleção 1 (NEW_SELECTION) - Grids que Intersectam Redes
arcpy.management.SelectLayerByLocation("layer_temp", overlap_type = "INTERSECT", select_features = redes, selection_type = "NEW_SELECTION")

# Seleção 2 (ADD_TO_SELECTION = OR) - Grids que Intersectam Ligações
arcpy.management.SelectLayerByLocation("layer_temp", overlap_type = "INTERSECT", select_features = ligacoes, selection_type = "ADD_TO_SELECTION")

# Exportar somente os selecionados
arcpy.management.CopyFeatures("layer_temp", output_fc)

# Excluindo do Contents as Camadas Temporárias Criadas
camadas_remover = ["layer_temp", "Grid_100m"]
for lyr in mapa.listLayers():
    if lyr.name in camadas_remover:
        mapa.removeLayer(lyr)

print("✔️ Processo concluído com sucesso!")

✔️ Processo concluído com sucesso!


In [8]:
# Selecionando Grids Dentro do DF (Excluindo Águas Lindas)

In [9]:
# Emtradas
input_fc = "Grid_Selecionado"
df = "Area_PAS"

# Criar layer (necessário para seleção)
arcpy.management.MakeFeatureLayer(input_fc, "layer_temp")

# Seleção - Grids dentro do DF
arcpy.management.SelectLayerByLocation("layer_temp", overlap_type = "INTERSECT", select_features = df, selection_type = "NEW_SELECTION")

# Inverter seleção 
arcpy.management.SelectLayerByAttribute("layer_temp", "SWITCH_SELECTION")

# Deletar os que não intersectam (Águas Lindas)
arcpy.management.DeleteFeatures("layer_temp")

for lyr in mapa.listLayers():
    if lyr.name == "layer_temp":
        mapa.removeLayer(lyr)
        break

print("✔️ Processo concluído com sucesso!")

✔️ Processo concluído com sucesso!


In [10]:
# Criar ID do Grid

In [11]:
grid = "Grid_Selecionado"

# Criar campo ID
arcpy.management.AddField(grid, "ID_Grid", "LONG")

print("Campo ID_Grid criado com sucesso.")

with arcpy.da.UpdateCursor(grid, ["ID_Grid"]) as cursor:    
    i = 1    
    for row in cursor:
        row[0] = i
        cursor.updateRow(row)
        i += 1

print("✅ ID_Grid preenchido sequencialmente.")

Campo ID_Grid criado com sucesso.
✅ ID_Grid preenchido sequencialmente.


In [12]:
# Intersect Redes com Grids

In [13]:
# Lista de Entrada do Intersect
input_features = ["Redes_SES_BDG", "Grid_Selecionado"]

# Output
output_fc = f"{gdb}\\{dataset}\\Redes_SES_Grid"

# Executar Intersect
arcpy.analysis.Intersect(
    in_features=input_features,
    out_feature_class=output_fc,
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="LINE"  
)

for lyr in mapa.listLayers():
    if lyr.name == "Redes_SES_BDG":
        mapa.removeLayer(lyr)
        break

print("✅ Intersect concluído com campos filtrados.")

✅ Intersect concluído com campos filtrados.


* INICIANDO ANÁLISE MULTI CRITÉRIO

ETAPA 1 - OS, LIGAÇÕES e Componentes

In [14]:
# Criar camada Temporária de OS de Ramal

In [15]:
# Nome da camada no BD
input_layer = "BDGIS.Ordens_Servico"

# Expressão SQL de filtro
sql_filter = """
DATAEXECUCAOINICIO between '2025-01-01 00:00:00' and '2025-12-31 23:59:59'
AND
MOTIVONAOEXECUCAO = 0
AND 
DESCSITUACAOOS = 'Baixada'
AND 
SERVAPROPRIADO in ('8102008021040', '8102008021070', '8102008021116', '8102008021115')
"""

# Criar uma feature layer temporária já filtrada
filtered_layer = "OS_Filtradas_Desobstrucao_25"
arcpy.MakeFeatureLayer_management(input_layer, filtered_layer, sql_filter)

<Result 'OS_Filtradas_Desobstrucao_25'>

In [16]:
# Criar Camada Temporária de Componentes

In [17]:
# Entrada
input_trecho = "Inspeção e Manutenção BDG"

sql_trecho = """
lifecyclestatus = 8 AND ASSETGROUP = 32
"""

layer_trecho = "Componentes_BDG"
arcpy.MakeFeatureLayer_management(input_trecho, layer_trecho, sql_trecho)

print("✅ Camada temporária criada: Componentes_BDG")

✅ Camada temporária criada: Componentes_BDG


In [11]:
# Exportar camada de Ligações

In [12]:
# Entrada
input_ligacoes = "Ligacoes_BDG"

out_path = os.path.join(gdb, dataset)
out_name = "Ligacoes_Limpo_Local"

# SQL de filtro
sql_ligacoes = """
BDGIS.TB_GCOMIMOVEIS.SITUACAOLIGACAOAGUADESC IN ('Ativa', 'Inativa')
AND BDGIS.TB_GCOMIMOVEIS.INSCRICAOAGRUPADORA IS NULL
"""

# Campos que queremos manter
campos_desejados = [
    "BDGIS.TB_GCOMIMOVEIS.INSCRICAO",
    "BDGIS.TB_GCOMIMOVEIS.CATEGORIADESC",
    "BDGIS.TB_GCOMIMOVEIS.TIPOAGRUPAMENTO",
    "BDGIS.TB_GCOMIMOVEIS.SITUACAOLIGACAOAGUADESC",
    "BDGIS.TB_GCOMIMOVEIS.INSCRICAOAGRUPADORA"
]

# FieldMappings
field_mappings = arcpy.FieldMappings()

for campo in campos_desejados:
    fm = arcpy.FieldMap()
    fm.addInputField(input_ligacoes, campo)
    field_mappings.addFieldMap(fm)

# Criar feature class enxuta no dataset
arcpy.FeatureClassToFeatureClass_conversion(
    in_features=input_ligacoes,
    out_path=out_path,
    out_name=out_name,
    where_clause=sql_ligacoes,
    field_mapping=field_mappings
)

print("✅ Feature class enxuta criada com sucesso no dataset Temporario_Counts")

✅ Feature class enxuta criada com sucesso no dataset Temporario_Counts


In [13]:
# Intersect Ligações com Grids

In [14]:
# Camadas de entrada do Intersect
input_features = ["Ligacoes_Limpo_Local", "Grid_Selecionado"]

# Nome da saída
output_name = "Ligacoes_Grids"

# Caminho completo da saída (GDB + Dataset + Feature Class)
output_fc = os.path.join(gdb, dataset, output_name)

# Executar Intersect
arcpy.analysis.Intersect(
    in_features=input_features,
    out_feature_class=output_fc,
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="POINT"  
)

# Removendo Camada Completa de Ligações
for lyr in mapa.listLayers():
    if lyr.name == "Ligacoes_Limpo_Local":
        mapa.removeLayer(lyr)
        break

print("✅ Intersect criado com sucesso e camada Ligacoes_Limpo_Local excluída do content!")

✅ Intersect criado com sucesso e camada Ligacoes_Limpo_Local excluída do content!


In [15]:
# Contando as Camadas Pontuais (Point) dentro dos Grids

In [16]:
# Caminhos
temp_gdb = os.path.join(gdb, "Temporario_Counts")

# Camada de Quadras 
poligonos = "Grid_Selecionado"

# Camada pontuais temporárias - Já filtradas
camadas_pontuais = {
    "Ligacoes_Grids": "Qtd_Ligacoes_Pressurizadas",
    "OS_Filtradas_Desobstrucao_25": "Qtd_OS_Desobstrucao",
    "Componentes_BDG": "Qtd_Componentes"
}

# Temporários
temp_layers = []
temp_layer_names = []

# Campo de ID Temporário 
temp_oid_field = "OID_Orig"
if temp_oid_field not in [f.name for f in arcpy.ListFields(poligonos)]:
    arcpy.management.AddField(poligonos, temp_oid_field, "LONG")
    with arcpy.da.UpdateCursor(poligonos, ["OID@", temp_oid_field]) as ucursor:
        for row in ucursor:
            row[1] = row[0]
            ucursor.updateRow(row)

# Contando camadas pontuais dentro do Polígono
for camada, campo_saida in camadas_pontuais.items():
    print(f"Processando {camada}...")
    
    out_join = os.path.join(temp_gdb, f"{camada}_join") # Salvando camada de saída no Dataset Temporario
    nome_join = f"{camada}_join" # Nome da camada de saída
    
    temp_layers.append(out_join) # Acrescentando na lista para exlcuir do Dataset Temporario - Se necessário
    temp_layer_names.append(nome_join) # Acrescentando na lista para excluir do Contents - Recomendado para não poluir o Contents

    arcpy.analysis.SpatialJoin(
        target_features=poligonos,
        join_features=camada,
        out_feature_class=out_join,
        join_operation="JOIN_ONE_TO_ONE",
        join_type="KEEP_ALL",
        match_option="INTERSECT"
    )

    if campo_saida not in [f.name for f in arcpy.ListFields(poligonos)]:
        arcpy.management.AddField(poligonos, campo_saida, "LONG")

    counts = {row[0]: row[1] for row in arcpy.da.SearchCursor(out_join, ["TARGET_FID", "Join_Count"])}

    with arcpy.da.UpdateCursor(poligonos, ["OID@", campo_saida]) as ucursor:
        for row in ucursor:
            row[1] = counts.get(row[0], 0)
            ucursor.updateRow(row)

# Limpeza - Recomendado
print("Excluindo do Contents as Camadas Criadas")
for lyr in mapa.listLayers():
    if lyr.name in temp_layer_names:
        mapa.removeLayer(lyr)

# Excluindo ID Temporário 
if temp_oid_field in [f.name for f in arcpy.ListFields(poligonos)]:
    arcpy.management.DeleteField(poligonos, temp_oid_field)

print("✅ Processo finalizado com sucesso!")

Processando Ligacoes_Grids...
Processando OS_Filtradas_Desobstrucao_25...
Processando Componentes_BDG...
Excluindo do Contents as Camadas Criadas
✅ Processo finalizado com sucesso!


In [18]:
# Criando Quebras Naturais para OS Desobstrução

In [19]:
# Entrada
fc = "Grid_Selecionado"
campo_valor = "Qtd_OS_Desobstrucao"
n_classes = 5

# Coletar valores (> 0)
valores = []

with arcpy.da.SearchCursor(fc, [campo_valor]) as cursor:
    for row in cursor:
        if row[0] > 0:
            valores.append(row[0])

# Calcular Jenks
jenks = mc.NaturalBreaks(valores, k=n_classes)

upper = jenks.bins  # limites superiores

print("Limites superiores:", upper)

# Calcular limites inferiores
lower = [min(valores)]  # primeiro limite inferior
lower += list(upper[:-1])

print("Limites inferiores:", lower)

# Mostrar intervalos
print("\nIntervalos das classes:")

for i, (l, u) in enumerate(zip(lower, upper), 1):
    print(f"Classe {i}: {l} < valor ≤ {u}")

Limites superiores: [  4.  10.  27.  65. 137.]
Limites inferiores: [1, 4.0, 10.0, 27.0, 65.0]

Intervalos das classes:
Classe 1: 1 < valor ≤ 4.0
Classe 2: 4.0 < valor ≤ 10.0
Classe 3: 10.0 < valor ≤ 27.0
Classe 4: 27.0 < valor ≤ 65.0
Classe 5: 65.0 < valor ≤ 137.0


In [20]:
# Calculando Nota Desobstrução

In [21]:
# Entrada
tabela = "Grid_Selecionado"
campo = "Nota_Desobstrucao"

# Criar campo
campos_existentes = [f.name for f in arcpy.ListFields(tabela)]
if campo not in campos_existentes:
    arcpy.management.AddField(tabela, campo, "SHORT")

# Expressão
expression = "classificar(!Qtd_OS_Desobstrucao!)"

# Quebras
b = lower

# Função dinâmica
codeblock = f"""
def classificar(valor):
    if valor == 0:
        return 0
    elif valor <= {b[0]}:
        return 1
    elif valor <= {b[1]}:
        return 2
    elif valor <= {b[2]}:
        return 3
    elif valor <= {b[3]}:
        return 4
    else:
        return 5
"""

# Calcular campo
arcpy.management.CalculateField(tabela, campo, expression, "PYTHON3", codeblock)

# Removendo Camada de Ligações
for lyr in mapa.listLayers():
    if lyr.name == "Ligacoes_Grids":
        mapa.removeLayer(lyr)
        break

print("✅ Nota Desobstrução calculada corretamente com Jenks!")

✅ Nota Desobstrução calculada corretamente com Jenks!


In [22]:
# Criando Quebras Naturais para Qtd Componentes

In [3]:
# Entrada
fc = "Grid_Selecionado"
campo_valor = "Qtd_Componentes"
n_classes = 5

# Coletar valores (> 0)
valores = []

with arcpy.da.SearchCursor(fc, [campo_valor]) as cursor:
    for row in cursor:
        if row[0] > 0:
            valores.append(row[0])

# Calcular Jenks
jenks = mc.NaturalBreaks(valores, k=n_classes)

upper = jenks.bins  # limites superiores

print("Limites superiores:", upper)

# Calcular limites inferiores
lower = [min(valores)]  # primeiro limite inferior
lower += list(upper[:-1])

print("Limites inferiores:", lower)

# Mostrar intervalos
print("\nIntervalos das classes:")

for i, (l, u) in enumerate(zip(lower, upper), 1):
    print(f"Classe {i}: {l} < valor ≤ {u}")

Limites superiores: [ 11.  25.  41.  60. 128.]
Limites inferiores: [1, 11.0, 25.0, 41.0, 60.0]

Intervalos das classes:
Classe 1: 1 < valor ≤ 11.0
Classe 2: 11.0 < valor ≤ 25.0
Classe 3: 25.0 < valor ≤ 41.0
Classe 4: 41.0 < valor ≤ 60.0
Classe 5: 60.0 < valor ≤ 128.0


In [4]:
# Calculando Nota de Densidade de Componentes

In [5]:
# Entrada
tabela = "Grid_Selecionado"
campo = "Nota_Componentes"

# Criar campo
campos_existentes = [f.name for f in arcpy.ListFields(tabela)]
if campo not in campos_existentes:
    arcpy.management.AddField(tabela, campo, "SHORT")

# Expressão
expression = "classificar(!Qtd_Componentes!)"

# Quebras
b = upper

# Função dinâmica
codeblock = f"""
def classificar(valor):
    if valor == 0:
        return 0
    elif valor <= {b[0]}:
        return 1
    elif valor <= {b[1]}:
        return 2
    elif valor <= {b[2]}:
        return 3
    elif valor <= {b[3]}:
        return 4
    else:
        return 5
"""

# Calcular campo
arcpy.management.CalculateField(tabela, campo, expression, "PYTHON3", codeblock)

# Removendo Camada de Ligações
for lyr in mapa.listLayers():
    if lyr.name == "Componentes_BDG":
        mapa.removeLayer(lyr)
        break

print("✅ Nota de componentes calculada corretamente com Jenks!")

✅ Nota de componentes calculada corretamente com Jenks!


ETAPA 2 - VOLUME MICROMEDIDO

In [12]:
# Tabela do Histórico de Leitura
table = r"Historico_Leitura"

# Colunas Desejadas
columns_final = [
    "REFERENCIA",
    "INSCRICAO",
    "VOLUMEMEDIDO",
    "SITUACAOLIGACAOAGUADESC",
    "INSCRICAOAGRUPADORA",
]

# Filtro Aplicado
where_clause = (
    "SITUACAOLIGACAOAGUADESC IN ('Ativa', 'Inativa') "
    "AND INSCRICAOAGRUPADORA IS NULL "
    "AND VOLUMEMEDIDO <> 0"
)

# Montando a estrutura do Data Frame 
rows = []
with arcpy.da.SearchCursor(
    table,
    columns_final,
    where_clause=where_clause
) as cursor:
    for row in cursor:
        rows.append(row)
    
# Convertendo pra Data Frame da Pandas
df_historico = pd.DataFrame(rows, columns=columns_final)

In [13]:
df_volume_medido = df_historico 

df_volume_medido.tail()

,REFERENCIA,INSCRICAO,VOLUMEMEDIDO,SITUACAOLIGACAOAGUADESC,INSCRICAOAGRUPADORA
17397446,202601,3886042,25,Ativa,None
17397447,202601,2281724,28,Ativa,None
17397448,202601,2227142,37,Ativa,None
17397449,202601,1394861,107,Ativa,None
17397450,202601,1524097,58,Ativa,None


In [14]:
table = r"Ligacoes_Grids"
columns = [f.name for f in arcpy.ListFields(table) if f.type!="Geometry"] #List the fields you want to include. I want all columns except the geometry#
df_ligacoes = pd.DataFrame(data=arcpy.da.SearchCursor(table, columns), columns=columns)

In [15]:
df_ligacoes.tail()

,OBJECTID,FID_Ligacoes_Limpo_Local,BDGIS_TB_GCOMIMOVEIS_INSCRICAO,BDGIS_TB_GCOMIMOVEIS_CATEGORIADESC,BDGIS_TB_GCOMIMOVEIS_TIPOAGRUPAMENTO,BDGIS_TB_GCOMIMOVEIS_SITUACAOLIGACAOAGUADESC,BDGIS_TB_GCOMIMOVEIS_INSCRICAOAGRUPADORA,FID_Grid_Selecionado,ID_Grid
380222,380223,651675,8740844,Residencial,Individual,Ativa,None,35921,18532
380223,380224,652917,8761108,Residencial,Individual,Ativa,None,35921,18532
380224,380225,652922,8761191,Residencial,Individual,Inativa,None,35921,18532
380225,380226,652923,8761205,Residencial,Individual,Inativa,None,35921,18532
380226,380227,655259,8796548,Residencial,Individual,Ativa,None,35921,18532


In [16]:
df_joined = pd.merge(
    df_ligacoes,
    df_volume_medido,
    left_on="BDGIS_TB_GCOMIMOVEIS_INSCRICAO",
    right_on="INSCRICAO",
    how="inner"  # opcional, mas é recomendado deixar explícito
)

In [17]:
df_joined.tail()

,OBJECTID,FID_Ligacoes_Limpo_Local,BDGIS_TB_GCOMIMOVEIS_INSCRICAO,BDGIS_TB_GCOMIMOVEIS_CATEGORIADESC,BDGIS_TB_GCOMIMOVEIS_TIPOAGRUPAMENTO,BDGIS_TB_GCOMIMOVEIS_SITUACAOLIGACAOAGUADESC,BDGIS_TB_GCOMIMOVEIS_INSCRICAOAGRUPADORA,FID_Grid_Selecionado,ID_Grid,REFERENCIA,INSCRICAO,VOLUMEMEDIDO,SITUACAOLIGACAOAGUADESC,INSCRICAOAGRUPADORA
9885246,380227,655259,8796548,Residencial,Individual,Ativa,None,35921,18532,202510,8796548,6,Ativa,None
9885247,380227,655259,8796548,Residencial,Individual,Ativa,None,35921,18532,202511,8796548,8,Ativa,None
9885248,380227,655259,8796548,Residencial,Individual,Ativa,None,35921,18532,202511,8796548,8,Ativa,None
9885249,380227,655259,8796548,Residencial,Individual,Ativa,None,35921,18532,202512,8796548,8,Ativa,None
9885250,380227,655259,8796548,Residencial,Individual,Ativa,None,35921,18532,202601,8796548,11,Ativa,None


In [18]:
# Calculando a Mediana de Volume Medido para cada Ligação - Considerando o Grid que a Ligação está!

In [19]:
df_mediana_volume = df_joined.groupby(by = ['INSCRICAO', 'ID_Grid'])['VOLUMEMEDIDO'].median().reset_index()

In [20]:
df_mediana_volume.tail()

,INSCRICAO,ID_Grid,VOLUMEMEDIDO
358805,9832149,4703,10.0
358806,9832157,4703,11.0
358807,9832513,3639,12.0
358808,9832661,17291,1.0
358809,9832751,18523,5.0


In [21]:
# Somando os Volumes dentro do Grid e Convertendo para L/s

In [30]:
# Calculando Volume por Grid
df_volume_grid = df_mediana_volume.groupby(by = ['ID_Grid'])['VOLUMEMEDIDO'].sum().reset_index()

# Convertendo de m³/mês para L/s
df_volume_grid['VOLUMEMEDIDO'] = df_volume_grid['VOLUMEMEDIDO'] / 2592

# Alterando Nome da Coluna
df_volume_grid.rename(columns={"VOLUMEMEDIDO": "Volume Medido Mediana (L/s)"}, inplace=True)

In [31]:
df_volume_grid.tail()

,ID_Grid,Volume Medido Mediana (L/s)
14794,18527,0.011188
14795,18528,0.002508
14796,18530,0.018904
14797,18531,0.012346
14798,18532,0.230517


In [32]:
# Exportando para Excel e trazendo novamento para o GIS

In [33]:
df_volume_grid.to_excel(r"C:\Projetos_ArcGis_Felipe_2025\Multi_Criterio_SES\Planilhas\Consumo_Grid.xlsx", index=False)

# Caminhos
excel_path = r"C:\Projetos_ArcGis_Felipe_2025\Multi_Criterio_SES\Planilhas\Consumo_Grid.xlsx"
output_name = "Tabela_Volume_Medido_Grid_Selecionado"

# Definir o caminho completo de saída
output_table = os.path.join(gdb, output_name)

# Nome da planilha dentro do Excel (geralmente "Sheet1$")
sheet_name = "Sheet1"

# Executar a ferramenta Excel To Table
arcpy.conversion.ExcelToTable(Input_Excel_File = excel_path, Output_Table = output_table, Sheet = sheet_name)

print(f"✅ Tabela criada com sucesso em: {output_table}")

✅ Tabela criada com sucesso em: C:\Projetos_ArcGis_Felipe_2025\Multi_Criterio_SES\Multi_Criterio_SES.gdb\Tabela_Volume_Medido_Grid_Selecionado


In [34]:
# Fazendo Join para ter os Quantitativos de Volume no Grid

In [21]:
# Entradas
input_layer = "Grid_Selecionado"
input_field = "ID_Grid"

# Tabela de join
join_table = "Tabela_Volume_Medido_Grid_Selecionado"
join_field = "ID_Grid"

# Saída
out_fc = os.path.join(gdb, dataset, "Grid_Final")

# Fazer o join
arcpy.AddJoin_management(
    in_layer_or_view=input_layer,
    in_field=input_field,
    join_table=join_table,
    join_field=join_field,
    join_type="KEEP_ALL"  # equivalente ao Left Join
)

print("✅ Join realizado com sucesso (temporário)")

# Exportar feature class com o join gravado
arcpy.FeatureClassToFeatureClass_conversion(
    in_features=input_layer,
    out_path=os.path.join(gdb, dataset),
    out_name="Grid_Final"
)

print("✅ Feature class exportada com join gravado")

# limpando dado após o Join
fields = [f.name for f in arcpy.ListFields(out_fc)]
campos_para_excluir = []

# OBJECTID herdado do join
if "OBJECTID" in fields:
    campos_para_excluir.append("OBJECTID")

# ID duplicado do join
if "ID_Grid_1" in fields:
    campos_para_excluir.append("ID_Grid_1")

if campos_para_excluir:
    arcpy.management.DeleteField(out_fc, campos_para_excluir)
    print(f"🧹 Campos excluídos: {campos_para_excluir}")

# Tratar NULLs para 0
campo_volume = "Volume_Medido_Mediana__m__mes_"

if campo_volume in [f.name for f in arcpy.ListFields(out_fc)]:
    with arcpy.da.UpdateCursor(out_fc, [campo_volume]) as ucursor:
        for row in ucursor:
            if row[0] is None:
                row[0] = 0
                ucursor.updateRow(row)

    print("🔢 Valores NULL convertidos para 0 no campo de volume")

# Removendo Camada de Ligações
for lyr in mapa.listLayers():
    if lyr.name == "Grid_Selecionado":
        mapa.removeLayer(lyr)
        break

print("✅ Processo finalizado com sucesso!")

✅ Join realizado com sucesso (temporário)
✅ Feature class exportada com join gravado
🧹 Campos excluídos: ['OBJECTID', 'ID_Grid_1']
🔢 Valores NULL convertidos para 0 no campo de volume
✅ Processo finalizado com sucesso!


In [22]:
# Calculando Quebras Naturais

In [41]:
# Entrada
fc = "Grid_Final"
campo_valor = "Volume_Medido_Mediana__L__s_"
n_classes = 5

# Coletar valores (> 0)
valores = []

with arcpy.da.SearchCursor(fc, [campo_valor]) as cursor:
    for row in cursor:
        if 0 < row[0] < 2:
            valores.append(row[0])

# Calcular Jenks
jenks = mc.NaturalBreaks(valores, k=n_classes)

upper = jenks.bins  # limites superiores

print("Limites superiores:", upper)

# Calcular limites inferiores
lower = [min(valores)]  # primeiro limite inferior
lower += list(upper[:-1])

print("Limites inferiores:", lower)

# Mostrar intervalos
print("\nIntervalos das classes:")

for i, (l, u) in enumerate(zip(lower, upper), 1):
    print(f"Classe {i}: {l} < valor ≤ {u}")

Limites superiores: [0.09587191 0.20968364 0.45987654 1.01099537 1.98533951]
Limites inferiores: [0.00038580246913580245, 0.09587191358024691, 0.20968364197530864, 0.45987654320987653, 1.0109953703703705]

Intervalos das classes:
Classe 1: 0.00038580246913580245 < valor ≤ 0.09587191358024691
Classe 2: 0.09587191358024691 < valor ≤ 0.20968364197530864
Classe 3: 0.20968364197530864 < valor ≤ 0.45987654320987653
Classe 4: 0.45987654320987653 < valor ≤ 1.0109953703703705
Classe 5: 1.0109953703703705 < valor ≤ 1.9853395061728396


In [24]:
# Nota Consumo

In [42]:
# Entrada
tabela = "Grid_Final"
campo = "Nota_Consumo"

# Criar campo
campos_existentes = [f.name for f in arcpy.ListFields(tabela)]
if campo not in campos_existentes:
    arcpy.management.AddField(tabela, campo, "SHORT")

# Expressão
expression = "classificar(!Volume_Medido_Mediana__L__s_!)"

# Quebras
b = upper

# Função dinâmica
codeblock = f"""
def classificar(valor):
    if valor == 0:
        return 0
    elif valor <= {b[0]}:
        return 1
    elif valor <= {b[1]}:
        return 2
    elif valor <= {b[2]}:
        return 3
    elif valor <= {b[3]}:
        return 4
    else:
        return 5
"""

# Calcular campo
arcpy.management.CalculateField(tabela, campo, expression, "PYTHON3", codeblock)

print("✅ Nota Consumo calculada corretamente com Jenks!")

✅ Nota Consumo calculada corretamente com Jenks!


* REDES

In [26]:
# Limpando Camada de Redes - Já previamente Intersectado com Grids

In [ ]:
'''
fc = "Redes_SES_Grid"

campos_manter = ["ID_Grid", "ASSETTYPE", "material", "diameter", "posicionamento", "dataimplantacao", "SHAPE_Length", "Extensao_Rede"]

# Listar campos da feature class
fields = arcpy.ListFields(fc)

campos_excluir = []

for f in fields:    
    if (f.name not in campos_manter and f.type not in ["OID", "Geometry"]):
        campos_excluir.append(f.name)

# Excluir campos indesejados
arcpy.management.DeleteField(fc, campos_excluir)

print("✅ Campos indesejados removidos com sucesso.")
'''

ETAPA 3 - NOTAS POR IDADE, MATERIAL, DIÂMETRO E DECLIVIDADE

* IDADE

In [28]:
# Camada de redes
redes = "Redes_SES_Grid"

# Criar campo de Idade
campo_idade = "Idade_Rede"

if campo_idade not in [f.name for f in arcpy.ListFields(redes)]:
    arcpy.management.AddField(redes, campo_idade, "LONG")

# Expressão
expression = """
calc(!dataimplantacao!)
"""

# Data de referência fixa (31/03/2025)
codeblock = """
from datetime import datetime

def calc(data_impl):
    if data_impl is None:
        return 0
    data_ref = datetime(2026, 4, 16)
    idade_dias = (data_ref - data_impl).days
    return idade_dias / 365.25
"""

# Calcular Idade
arcpy.management.CalculateField(redes, campo_idade, expression, "PYTHON3", codeblock)

print("✅ Idade da rede calculada (em anos).")

✅ Idade da rede calculada (em anos).


In [29]:
# Corrigindo as Datas Erradas

In [30]:
# Camada
redes = "Redes_SES_Grid"

# Criar layer temporária
redes_lyr = "redes_lyr"
arcpy.management.MakeFeatureLayer(redes, redes_lyr)

# Corrigir datas inválidas (idade = 31)
arcpy.management.SelectLayerByAttribute(redes_lyr, "NEW_SELECTION",
    """
    dataimplantacao IS NULL OR 
    dataimplantacao = timestamp '1900-01-01 00:00:00' OR 
    dataimplantacao = timestamp '1899-12-31 21:00:00' OR
    dataimplantacao = timestamp '31/12/1889 00:00:00'
    """
)

arcpy.management.CalculateField(redes_lyr, "Idade_Rede", "27", "PYTHON3")

# Corrigir idades negativas (idade = 1) 
arcpy.management.SelectLayerByAttribute(redes_lyr, "NEW_SELECTION", "Idade_Rede < 0")

arcpy.management.CalculateField(redes_lyr, "Idade_Rede", "1", "PYTHON3")

# Limpar seleção
arcpy.management.SelectLayerByAttribute(redes_lyr, "CLEAR_SELECTION")

# (opcional) Deletar layer
arcpy.management.Delete(redes_lyr)

print("✅ Idades inválidas e negativas corrigidas.")

✅ Idades inválidas e negativas corrigidas.


In [31]:
# Calcular campo Idade * Comprimento 

In [32]:
# Entrada
redes = "Redes_SES_Grid"

# Campo produto
campo_prod = "Idade_x_Comp"

if campo_prod not in [f.name for f in arcpy.ListFields(redes)]:
    arcpy.management.AddField(redes, campo_prod, "DOUBLE")

# Calcular produto
arcpy.management.CalculateField(redes, campo_prod,  "!Idade_Rede! * !SHAPE_Length!", "PYTHON3")

print("✅ Campo Idade x Comprimento calculado.")

✅ Campo Idade x Comprimento calculado.


In [33]:
# Somando Idade * Comprimento por Grid e Calculando Média Ponderada por Grid

In [34]:
# Entrada
redes = "Redes_SES_Grid"

# Saída
stats_idade = f"{gdb}\\Stats_Idade_Grid"

# Campo Média Ponderada por Grid
campo_media = "Idade_Media_Grid"

arcpy.analysis.Statistics(redes, stats_idade, [["Idade_x_Comp", "SUM"], ["SHAPE_Length", "SUM"]], "ID_Grid")

print("✅ Statistics da idade por grid calculado.")

# Criando Campo Idade Média por Grid
if campo_media not in [f.name for f in arcpy.ListFields(stats_idade)]:
    arcpy.management.AddField(stats_idade, campo_media, "DOUBLE")

arcpy.management.CalculateField(stats_idade, campo_media, "!SUM_Idade_x_Comp! / !SUM_SHAPE_Length!", "PYTHON3")

print("✅ Idade média por grid calculada.")

✅ Statistics da idade por grid calculado.
✅ Idade média por grid calculada.


In [35]:
# Atribuindo Nota por Idade

In [36]:
# Tabela
stats_idade = "Stats_Idade_Grid"

# Campo de nota
campo_nota = "Nota_Idade"

# Criando Campo Nota_Idade
if campo_nota not in [f.name for f in arcpy.ListFields(stats_idade)]:
    arcpy.management.AddField(stats_idade, campo_nota, "SHORT")

# Expressão
expression = """
calc(!Idade_Media_Grid!)
"""

codeblock = """
def calc(x):
    if x < 20.001:
        return 1
    elif x < 30.001:
        return 2
    elif x < 40.001:
        return 3
    elif x < 50.001:
        return 4
    else:
        return 5
"""

# Calcular Nota
arcpy.management.CalculateField(stats_idade, campo_nota, expression, "PYTHON3", codeblock)

print("✅ Nota por idade atribuída.")

✅ Nota por idade atribuída.


In [37]:
# Join da Tabela com a Camada de Grids

In [38]:
grid = "Grid_Final"
stats_idade = "Stats_Idade_Grid"

arcpy.management.JoinField(grid, "ID_Grid", stats_idade, "ID_Grid", ["Idade_Media_Grid", "Nota_Idade"])

print("✅ Percentual adicionado ao grid.")

# Substituir NULL por 0 após o Join
arcpy.management.CalculateField(grid, "Nota_Idade", "0 if !Nota_Idade! == None else !Nota_Idade!", "PYTHON3")

print("✅ NULLs trocados por 0.")

✅ Percentual adicionado ao grid.
✅ NULLs trocados por 0.


* MATERIAL

In [39]:
redes = "Redes_SES_Grid"

# Criar campo (se ainda não existir)
field_name = "Nota_Material"

if field_name not in [f.name for f in arcpy.ListFields(redes)]:
    arcpy.AddField_management(redes, field_name, "SHORT")

with arcpy.da.UpdateCursor(redes, ["material", "Nota_Material"]) as cursor:
    for material, nota in cursor:
        if "PEAD" in material:
            nota_final = 1
        elif "PVC" in material:
            nota_final = 2
        elif material in ["FF", "FV", "PRFV"]:
            nota_final = 3
        elif material == "MBV":
            nota_final = 4
        else:
            nota_final = 5
        
        cursor.updateRow([material, nota_final])

# Campo produto
campo_prod = "Material_x_Comp"

if campo_prod not in [f.name for f in arcpy.ListFields(redes)]:
    arcpy.management.AddField(redes, campo_prod, "DOUBLE")

# Calcular produto
arcpy.management.CalculateField(redes, campo_prod, "!Nota_Material! * !SHAPE_Length!", "PYTHON3")

print("✅ Notas atribuidas e campo Material_x_Comp calculado.")

<Result 'Redes_SES_Grid'>

In [40]:
# Agrupando por Grid e Calculando Média Ponderada por Comprimento

In [41]:
# Saída
stats_material = f"{gdb}\\Stats_Material_Grid"

# Campo Média Ponderada por Grid
campo_media = "Nota_Material"

arcpy.analysis.Statistics(redes, stats_material, [["Material_x_Comp", "SUM"], ["SHAPE_Length", "SUM"]], "ID_Grid")

print("✅ Statistics de material por grid calculado.")

# Criando Campo Nota Média Material por Grid
if campo_media not in [f.name for f in arcpy.ListFields(stats_material)]:
    arcpy.management.AddField(stats_material, campo_media, "DOUBLE")

arcpy.management.CalculateField(stats_material, campo_media, "!SUM_Material_x_Comp! / !SUM_Shape_Length!", "PYTHON3")

print("✅ Média Ponderada por grid calculada.")

✅ Statistics da material por grid calculado.
✅ Média Ponderada por grid calculada.


In [42]:
# Join da Tabela com a Camada de Grids

In [43]:
grid = "Grid_Final"
stats_material = "Stats_Material_Grid"

arcpy.management.JoinField(grid, "ID_Grid", stats_material, "ID_Grid", ["Nota_Material"])

print("✅ Nota adicionada grid.")

# Substituir NULL por 0 após o Join
arcpy.management.CalculateField(grid, "Nota_Material", "0 if !Nota_Material! == None else !Nota_Material!", "PYTHON3")

print("✅ NULLs trocados por 0.")

✅ Nota adicionada grid.
✅ NULLs trocados por 0.


* DIÂMETRO

In [44]:
redes = "Redes_SES_Grid"

# Criar campo (se ainda não existir)
field_name = "Nota_Diametro"

if field_name not in [f.name for f in arcpy.ListFields(redes)]:
    arcpy.AddField_management(redes, field_name, "SHORT")

with arcpy.da.UpdateCursor(redes, ["diameter", "Nota_Diametro"]) as cursor:
    for diametro, nota in cursor:
        if diametro < 101:
            nota = 5
        elif diametro < 151:
            nota = 4
        elif diametro < 201:
            nota = 3
        elif diametro < 301:
            nota = 2
        else:
            nota = 1
        
        cursor.updateRow([diametro, nota])

# Campo produto
campo_prod = "Diametro_x_Comp"

if campo_prod not in [f.name for f in arcpy.ListFields(redes)]:
    arcpy.management.AddField(redes, campo_prod, "DOUBLE")

# Calcular produto
arcpy.management.CalculateField(redes, campo_prod, "!Nota_Diametro! * !SHAPE_Length!", "PYTHON3")

print("✅ Notas atribuidas e campo Diametro_x_Comp calculado.")

<Result 'Redes_SES_Grid'>

In [35]:
# Agrupando por Grid e Calculando Média Ponderada por Comprimento

In [46]:
# Saída
stats_diametro = f"{gdb}\\Stats_Diametro_Grid"

# Campo Média Ponderada por Grid
campo_media = "Nota_Diametro"

arcpy.analysis.Statistics(redes, stats_diametro, [["Diametro_x_Comp", "SUM"], ["SHAPE_Length", "SUM"]], "ID_Grid")

print("✅ Statistics de diâmetro por grid calculado.")

# Criando Campo Nota Média Diâmetro por Grid
if campo_media not in [f.name for f in arcpy.ListFields(stats_diametro)]:
    arcpy.management.AddField(stats_diametro, campo_media, "DOUBLE")

arcpy.management.CalculateField(stats_diametro, campo_media, "!SUM_Diametro_x_Comp! / !SUM_Shape_Length!", "PYTHON3")

print("✅ Média Ponderada por grid calculada.")

✅ Statistics de diâmetro por grid calculado.
✅ Média Ponderada por grid calculada.


In [47]:
# Join da Tabela com a Camada de Grids

In [48]:
grid = "Grid_Final"
stats_material = "Stats_Diametro_Grid"

arcpy.management.JoinField(grid, "ID_Grid", stats_material, "ID_Grid", ["Nota_Diametro"])

print("✅ Nota adicionada ao grid.")

# Substituir NULL por 0 após o Join
arcpy.management.CalculateField(grid, "Nota_Diametro", "0 if !Nota_Diametro! == None else !Nota_Diametro!", "PYTHON3")

print("✅ NULLs trocados por 0.")

✅ Nota adicionada ao grid.
✅ NULLs trocados por 0.


* DECLIVIDADE

In [2]:
# Criando nova coluna para análise e ajustando valores nulos, iguais a zero e -1

In [3]:
# Emtrada
redes = "Redes_SES_Grid"

# Nova Coluna
campo_decliv = "Declividade_Analise"

fields = [f.name for f in arcpy.ListFields(redes)]
if campo_decliv not in fields:
    arcpy.management.AddField(redes, campo_decliv, "DOUBLE")

# Ajuste de Valores
with arcpy.da.UpdateCursor(redes, ["slope", campo_decliv]) as cursor:
    for slope, decliv in cursor:
        
        # Declividade mínima normativa
        decliv_min = 0.005
        
        if slope is None:
            decliv_final = 0.0159
        
        elif slope == 0:
            decliv_final = 0.0159

        elif slope == -1:
            decliv_final = 0.0159  

        elif slope > 0.1:
            decliv_final = 0.0159 

        elif slope < 0:
            decliv_final = decliv_min
                
        else:
            decliv_final = slope
        
        cursor.updateRow([slope, decliv_final])

print("✅ Nova coluna criada e valores de declividade ajustados.")

✅ Nova coluna criada e valores de declividade ajustados.


In [4]:
# Calculando Nota por trecho e multiplicando pelo Comprimento

In [5]:
redes = "Redes_SES_Grid"

# Criar campo (se ainda não existir)
field_name = "Nota_Declividade"

if field_name not in [f.name for f in arcpy.ListFields(redes)]:
    arcpy.AddField_management(redes, field_name, "SHORT")

with arcpy.da.UpdateCursor(redes, ["Declividade_Analise", "Nota_Declividade"]) as cursor:
    for declividade, nota in cursor:
        if declividade <= 0.005:
            nota = 5
        elif declividade <= 0.010:
            nota = 4
        elif declividade <= 0.020:
            nota = 1
        elif declividade <= 0.050:
            nota = 2
        else:
            nota = 3
        
        cursor.updateRow([declividade, nota])

# Campo Produto
campo_prod = "Decliv_x_Comp"

if campo_prod not in [f.name for f in arcpy.ListFields(redes)]:
    arcpy.management.AddField(redes, campo_prod, "DOUBLE")

# Calcular Produto
arcpy.management.CalculateField(redes, campo_prod, "!Nota_Declividade! * !SHAPE_Length!", "PYTHON3")

print("✅ Campo Declividade x Comprimento calculado.")

✅ Campo Declividade x Comprimento calculado.


In [6]:
# Somando Nota Declividade * Comprimento por Grid e Calculando Média Ponderada por Grid

In [7]:
# Entrada
redes = "Redes_SES_Grid"

# Saída
stats_declividade = f"{gdb}\\Stats_Declividade_Grid"

# Campo Média Ponderada por Grid
campo_media = "Nota_Declividade"

arcpy.analysis.Statistics(redes, stats_declividade, [["Decliv_x_Comp", "SUM"], ["SHAPE_Length", "SUM"]], "ID_Grid")

print("✅ Statistics da Declividade por grid calculado.")

# Criando Campo Declividade Média por Grid
if campo_media not in [f.name for f in arcpy.ListFields(stats_declividade)]:
    arcpy.management.AddField(stats_declividade, campo_media, "DOUBLE")

arcpy.management.CalculateField(stats_declividade, campo_media, "!SUM_Decliv_x_Comp! / !SUM_SHAPE_Length!", "PYTHON3")

print("✅ Declividade média por grid calculada.")

✅ Statistics da Declividade por grid calculado.
✅ Declividade média por grid calculada.


In [8]:
# Join da Tabela com a Camada de Grids

In [9]:
grid = "Grid_Final"
stats_declividade = "Stats_Declividade_Grid"

arcpy.management.JoinField(grid, "ID_Grid", stats_declividade, "ID_Grid", ["Nota_Declividade"])

print("✅ Nota adicionada ao grid.")

# Substituir NULL por 0 após o Join
arcpy.management.CalculateField(grid, "Nota_Declividade", "0 if !Nota_Declividade! == None else !Nota_Declividade!", "PYTHON3")

print("✅ NULLs trocados por 0.")

✅ Nota adicionada ao grid.
✅ NULLs trocados por 0.


* RESULTADO FINAL

In [2]:
# Camada do Grid (onde estão todas as notas)
grid = "Grid_Final"

# Criar campo resultado
campo_resultado = "Resultado"

if campo_resultado not in [f.name for f in arcpy.ListFields(grid)]:
    arcpy.management.AddField(grid, campo_resultado, "DOUBLE")

# Expressão
expression = """
calc(!Nota_Consumo!, !Nota_Declividade!, !Nota_Desobstrucao!, !Nota_Idade!, !Nota_Diametro!, !Nota_Material!, !Nota_Componentes!)
"""

codeblock = """
def calc(consumo, declividade, desobstrucao, idade, diametro, material, componentes):
    try:
        return (
            (consumo * 0.15) +
            (declividade * 0.20) + 
            (desobstrucao * 0.15) +
            (idade * 0.15) +
            (diametro * 0.20) +
            (material * 0.10) +
            (componentes * 0.05) 
        )
    except:
        return 0
"""

# Calcular resultado final
arcpy.management.CalculateField(grid, campo_resultado, expression, "PYTHON3", codeblock)

print("✅ Resultado final calculado.")

✅ Resultado final calculado.


* IDW

In [5]:
# Gerando Camada de Pontos para Grids

In [6]:
# Entrada
entrada = "Grid_Final"

# Saída
saida = os.path.join(gdb, dataset, "Pontos_Grid_Final")

# Criar pontos (centroides)
arcpy.management.FeatureToPoint(in_features = entrada, out_feature_class = saida)

print("✅ Pontos criados a partir dos grids.")

✅ Pontos criados a partir dos grids.


In [7]:
# Executando IDW

In [8]:
# Ativar extensão Spatial Analyst
arcpy.CheckOutExtension("Spatial")

# Entrada
pontos = "Pontos_Grid_Final"

# Saída
idw_saida = f"{gdb}\\IDW"

# Parâmetros
campo = "Resultado"
cell_size = 5

# Raio variável (12 pontos, distância máxima 150 m)
search_radius = RadiusVariable(12, 150)

# Executar IDW
IDW = Idw(
    in_point_features=pontos,
    z_field=campo,
    cell_size=cell_size,
    power=2,
    search_radius=search_radius
)

# Salvar resultado
IDW.save(idw_saida)

# Removendo Camada de Pontos
for lyr in mapa.listLayers():
    if lyr.name == "Pontos_Grid_Final":
        mapa.removeLayer(lyr)
        break

print("✅ IDW gerado com sucesso com raio variável.")

✅ IDW gerado com sucesso com raio variável.


In [8]:
# Cortando IDW

In [9]:
# Raster de entrada
input_raster = "IDW"

# Impedir Raster Out_Extract de ser adicionado ao mapa!
arcpy.env.addOutputsToMap = False

# Feature class
aoi_fc = "AOI_Sul_Oeste_v4"

# Campo com nomes
name_field = "RA_Python_IDW"

# Lista para armazenar rasters gerados
rasters_gerados = []

# Cortando Rasters por Área de Interesse
with arcpy.da.SearchCursor(aoi_fc, ["SHAPE@", name_field]) as cursor:
    for geom, nome in cursor:
        
        # Saída
        out_raster = os.path.join(gdb_rasters, nome)

        # Executar recorte
        out_extract = ExtractByMask(input_raster, geom)

        # Salvar raster final
        out_extract.save(out_raster)

        # Guardar caminho
        rasters_gerados.append((out_raster, nome))  # guarda também o nome

print("✅ Rasters gerados com sucesso.")

# Adicionar ao mapa com nome correto
for raster, nome in rasters_gerados:
    layer = mapa.addDataFromPath(raster)
    layer.name = nome  # Resolve o problema do "out_extract"

print("✅ Rasters adicionados ao Contents com nome correto.")

✅ Rasters gerados com sucesso.
✅ Rasters adicionados ao Contents com nome correto.


In [10]:
# Simbologia - Aplicando Simbologia a partir de camadas já Prontas

In [11]:
# Pasta onde estão salvos os arquivos .lyrx
lyrx_folder = r"F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Lyrx_IDW_SES"

# Mapeamento: RA -> arquivo base de simbologia (.lyrx)
mapping = {
    "IDW_Aguas_Claras": "IDW_Aguas_Claras",
    "IDW_Arniqueira": "IDW_Arniqueira",
    "IDW_Brazlandia": "IDW_Brazlandia",
    "IDW_Ceilandia": "IDW_Ceilandia",
    "IDW_Gama": "IDW_Gama",
    "IDW_Incra_8": "IDW_Incra_8",
    "IDW_Recanto_das_Emas": "IDW_Recanto_das_Emas",
    "IDW_Riacho_Fundo": "IDW_Riacho_Fundo",
    "IDW_Riacho_Fundo_II": "IDW_Riacho_Fundo_II",
    "IDW_Samambaia": "IDW_Samambaia",
    "IDW_Santa_Maria": "IDW_Santa_Maria",
    "IDW_Sol_Nascente_e_Por_do_Sol": "IDW_Sol_Nascente_e_Por_do_Sol",
    "IDW_Taguatinga": "IDW_Taguatinga",
    "IDW_Area_Sul": "IDW_Area_Sul",
    "IDW_Area_Oeste": "IDW_Area_Oeste"
}

# Aplicando simbologia para todos os rasters

# Aplicando simbologia
for lyr in mapa.listLayers():
    
    # Garantir que é raster
    if not lyr.isRasterLayer:
        continue
    
    nome_layer = lyr.name
    
    # Caminho direto do lyrx com mesmo nome
    symbology_file = os.path.join(lyrx_folder, nome_layer + ".lyrx")
    
    # Verificar se o arquivo existe (boa prática)
    if os.path.exists(symbology_file):
        arcpy.management.ApplySymbologyFromLayer(lyr, symbology_file)
    else:
        print(f"⚠️ Lyrx não encontrado para: {nome_layer}")

print("🎨 Processo de simbologia finalizado.")

🎨 Processo de simbologia finalizado.


In [13]:
# Aplicando Simbologia por Quebras Naturais

In [27]:
for lyr in mapa.listLayers():
    
    # Filtrar apenas rasters que começam com IDW
    if lyr.isRasterLayer and lyr.name.startswith("IDW"):
        
        print(f"Aplicando simbologia em: {lyr.name}")
        
        sym = lyr.symbology
        
        # Garantir que é do tipo raster stretch/classify
        if hasattr(sym, "colorizer"):
            
            sym.updateColorizer('RasterClassifyColorizer')
            
            # Classificação
            sym.colorizer.classificationMethod = "NaturalBreaks"
            sym.colorizer.breakCount = 5
            
            # Esquema de cores
            sym.colorizer.colorRamp = aprx.listColorRamps("Condition Number")[0]
            
            # Aplicar simbologia de volta
            lyr.symbology = sym
        
        # Transparência
        lyr.transparency = 30

print("✅ Simbologia aplicada a todos os rasters IDW.")

Aplicando simbologia em: IDW_Area_Sul
Aplicando simbologia em: IDW_Area_Oeste
Aplicando simbologia em: IDW_Incra_8
Aplicando simbologia em: IDW_Arniqueira
Aplicando simbologia em: IDW_Sol_Nascente_e_Por_do_Sol
Aplicando simbologia em: IDW_Aguas_Claras
Aplicando simbologia em: IDW_Taguatinga
Aplicando simbologia em: IDW_Brazlandia
Aplicando simbologia em: IDW_Gama
Aplicando simbologia em: IDW_Recanto_das_Emas
Aplicando simbologia em: IDW_Santa_Maria
Aplicando simbologia em: IDW_Riacho_Fundo_II
Aplicando simbologia em: IDW_Riacho_Fundo
Aplicando simbologia em: IDW_Samambaia
Aplicando simbologia em: IDW_Ceilandia
Aplicando simbologia em: IDW
✅ Simbologia aplicada a todos os rasters IDW.


* EXPORTANDO LAYOUTS

In [4]:
# RAs

In [12]:
# Caminho do projeto .aprx
aprx_path = r"F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Multi_Criterio_SES.aprx"
aprx = arcpy.mp.ArcGISProject(aprx_path)

# Nome do layout modelo
layout_model = aprx.listLayouts("A1_MULTI_CRITERIO")[0]  # ajuste se necessário

# Pasta de saída
out_folder = r"F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Mapas_Multi_Criterio_SES"

# Mapa principal (rasters + urbanismo)
mapa_principal = aprx.listMaps("Mapa_Principal")[0]

# Map frame do layout
map_frame = layout_model.listElements("MAPFRAME_ELEMENT", "Map_Frame_Principal")[0]

# Elemento de texto do selo (ajuste o nome conforme definido no layout)
selo_elemento = layout_model.listElements("TEXT_ELEMENT", "Texto_Selo")[0]

# Bookmarks do mapa principal
bookmarks = mapa_principal.listBookmarks()

# Nomes das camadas de urbanismo
camadas_urbanismo = ["Via_por_AOI_v4", "Quadras_Label"]

for bm in bookmarks:
    nome = bm.name

    # Desligar todas as camadas
    for lyr in mapa_principal.listLayers():
        lyr.visible = False

    # Ligar apenas o raster correspondente ao bookmark
    camada = [lyr for lyr in mapa_principal.listLayers() if lyr.name == nome]
    if not camada:
        print(f"⚠ Nenhuma camada encontrada para {nome}")
        continue
    camada[0].visible = True

    selo_texto = nome  # valor padrão caso não ache na tabela

    # Ativar camadas de urbanismo com filtro RA_Python_IDW
    for urb_name in camadas_urbanismo:
        urb_layer = [lyr for lyr in mapa_principal.listLayers() if lyr.name == urb_name]
        if urb_layer:
            urb_layer = urb_layer[0]
            urb_layer.visible = True
            urb_layer.definitionQuery = f"RA_Python_IDW = '{nome}'"

            # Se for a camada Via_por_AOI_v4_Sul_Oeste, buscar o valor do campo Selo
            if urb_name == "Via_por_AOI_v4":
                with arcpy.da.SearchCursor(urb_layer.dataSource, ["RA_Python_IDW", "Selo"]) as cursor:
                    for row in cursor:
                        if row[0] == nome:
                            selo_texto = row[1]
                            break
        else:
            print(f"⚠ Camada de urbanismo {urb_name} não encontrada no mapa")

    # Atualizar selo no layout
    selo_elemento.text = selo_texto

    # Aplicar o bookmark no map frame
    map_frame.zoomToBookmark(bm)

    # Exportar PNG
    out_png = os.path.join(out_folder, f"{nome}.png")
    layout_model.exportToPNG(out_png, resolution=300)
    print(f"✅ Layout exportado: {out_png} (Selo: {selo_texto})")

✅ Layout exportado: F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Mapas_Multi_Criterio_SES\IDW_Gama.png (Selo: GAMA - RA-II)
✅ Layout exportado: F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Mapas_Multi_Criterio_SES\IDW_Aguas_Claras.png (Selo: ÁGUAS CLARAS - RA-XX)
✅ Layout exportado: F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Mapas_Multi_Criterio_SES\IDW_Taguatinga.png (Selo: TAGUATINGA - RA-III)
✅ Layout exportado: F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Mapas_Multi_Criterio_SES\IDW_Brazlandia.png (Selo: BRAZLÂNDIA - RA-IV)
✅ Layout exportado: F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Mapas_Multi_Criterio_SES\IDW_Samambaia.png (Selo: SAMAMBAIA - RA-XII)
✅ Layout exportado: F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Mapas_Multi_Criterio_SES\IDW_Sol_Nascente_e_Por_do_Sol.png (Selo: SOL NASCENTE E POR DO SOL - RA-XXXII)
✅ Layout exportado: F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Mapas_Multi_Criterio_SES\IDW_Ceilandia.png (Selo: CEILÂNDIA - RA-IX)
✅ Layout exportado: F:\Projetos_ArcGIS_

In [12]:
# Área Sul e Oeste - Ainda é necessário apagar os bookmarks!

In [13]:
# Caminho do projeto .aprx
aprx_path = r"F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Multi_Criterio_SES.aprx"
aprx = arcpy.mp.ArcGISProject(aprx_path)

# Nome do layout modelo
layout_model = aprx.listLayouts("A1_MULTI_CRITERIO")[0]  # ajuste se necessário

# Pasta de saída
out_folder = r"F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Mapas_Multi_Criterio_SES"

# Mapa principal (rasters + urbanismo)
mapa_principal = aprx.listMaps("Mapa_Principal")[0]

# Map frame do layout
map_frame = layout_model.listElements("MAPFRAME_ELEMENT", "Map_Frame_Principal")[0]

# Elemento de texto do selo (ajuste o nome conforme definido no layout)
selo_elemento = layout_model.listElements("TEXT_ELEMENT", "Texto_Selo")[0]

# Bookmarks do mapa principal
bookmarks = mapa_principal.listBookmarks()

# Nomes das camadas de urbanismo
camadas_urbanismo = ["Via_por_AOI_v4_Sul_Oeste", "Quadras_Label_Mapas_de_Calor_Sul_Oeste"]

for bm in bookmarks:
    nome = bm.name

    # Desligar todas as camadas
    for lyr in mapa_principal.listLayers():
        lyr.visible = False

    # Ligar apenas o raster correspondente ao bookmark
    camada = [lyr for lyr in mapa_principal.listLayers() if lyr.name == nome]
    if not camada:
        print(f"⚠ Nenhuma camada encontrada para {nome}")
        continue
    camada[0].visible = True

    selo_texto = nome  # valor padrão caso não ache na tabela

    # Ativar camadas de urbanismo com filtro RA_Python_IDW
    for urb_name in camadas_urbanismo:
        urb_layer = [lyr for lyr in mapa_principal.listLayers() if lyr.name == urb_name]
        if urb_layer:
            urb_layer = urb_layer[0]
            urb_layer.visible = True
            urb_layer.definitionQuery = f"RA_Python_IDW = '{nome}'"

            # Se for a camada Via_por_AOI_v4_Sul_Oeste, buscar o valor do campo Selo
            if urb_name == "Via_por_AOI_v4_Sul_Oeste":
                with arcpy.da.SearchCursor(urb_layer.dataSource, ["RA_Python_IDW", "Selo"]) as cursor:
                    for row in cursor:
                        if row[0] == nome:
                            selo_texto = row[1]
                            break
        else:
            print(f"⚠ Camada de urbanismo {urb_name} não encontrada no mapa")

    # Atualizar selo no layout
    selo_elemento.text = selo_texto

    # Aplicar o bookmark no map frame
    map_frame.zoomToBookmark(bm)

    # Exportar PNG
    out_png = os.path.join(out_folder, f"{nome}.png")
    layout_model.exportToPNG(out_png, resolution=700)
    print(f"✅ Layout exportado: {out_png} (Selo: {selo_texto})")

✅ Layout exportado: F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Mapas_Multi_Criterio_SES\IDW_Area_Sul.png (Selo: ÁREA SUL)
✅ Layout exportado: F:\Projetos_ArcGIS_Gama\Multi_Criterio_SES\Mapas_Multi_Criterio_SES\IDW_Area_Oeste.png (Selo: ÁREA OESTE)


* DEFININDO ÁREAS PRIORITÁRIAS

In [ ]:
# Quebras Naturais do IDW

In [ ]:
# Raster de entrada
raster = "IDW"

# Converter raster para array
arr = arcpy.RasterToNumPyArray(raster, nodata_to_value=np.nan)

# Flatten (transformar em vetor 1D)
valores = arr.flatten()

# Remover NoData e zeros (opcional)
valores = valores[~np.isnan(valores)]
valores = valores[valores > 0]

# Definir número de classes
n_classes = 5

# Calcular Jenks
jenks = mc.NaturalBreaks(valores, k=n_classes)

# Limites superiores
upper = jenks.bins

# Limites inferiores
lower = [valores.min()]
lower += list(upper[:-1])

# Mostrar resultados
print("Limites inferiores:", lower)
print("Limites superiores:", upper)

print("\nIntervalos:")
for i, (l, u) in enumerate(zip(lower, upper), 1):
    print(f"Classe {i}: {l:.2f} < valor ≤ {u:.2f}")

In [ ]:
# Reclassificando Raster

In [ ]:
# Importando Spatial Analysis
from arcpy.sa import *

# Entrada
idw = "IDW"

# Saída raster binário
raster_filtrado = f"{gdb}\\IDW_Areas_Prioritarias"

# Definir limite (início da última classe)
limite = upper[-2]

# Criar raster binário (1 = classe alta, NoData = resto)
IDW_Areas_Prioritarias = Con(Raster(idw) >= limite, 1)

IDW_Areas_Prioritarias.save(raster_filtrado)

print(f"✅ Raster da classe alta gerado (limite = {limite:.2f})")

In [ ]:
# Raster para Poly

In [ ]:
# Entradas
raster = "IDW_Areas_Prioritarias"

# Saída
saida_poligono = f"{gdb}\\{dataset}\\Areas_Prioritarias"

arcpy.conversion.RasterToPolygon(
    in_raster=raster,
    out_polygon_features=saida_poligono,
    simplify="SIMPLIFY",
    raster_field="Value"
)

print("✅ Polígono gerado com sucesso")

In [ ]:
# Calculando Quantidade de OS por Poly

In [ ]:
# Entradas
poligonos = "Areas_Prioritarias"
pontos = "OS_Filtradas_Ramal_25"

# Saída
saida = f"{gdb}\\{dataset}\\Areas_Prioritarias_Count_OS"

# Criar FieldMappings
field_mappings = arcpy.FieldMappings()
field_mappings.addTable(poligonos)  # Mantém apenas campos dos polígonos

# Spatial Join
arcpy.analysis.SpatialJoin(
    target_features=poligonos,
    join_features=pontos,
    out_feature_class=saida,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    field_mapping=field_mappings,
    match_option="INTERSECT"
)

# Removendo Camada de Ligações
for lyr in mapa.listLayers():
    if lyr.name == "Areas_Prioritarias":
        mapa.removeLayer(lyr)
        break

print("✅ Contagem de pontos por polígono concluída!")

In [ ]:
# Intersect com RAs

In [ ]:
# Entradas
areas_prioritarias = "Areas_Prioritarias_Count_OS"
ras = "AOI_Sul_Oeste_v4"

# Saída
saida = f"{gdb}\\{dataset}\\Areas_Prioritarias_Ranking_RAs"

# Executar Intersect
arcpy.analysis.Intersect(
    in_features=[areas_prioritarias, ras],
    out_feature_class=saida,
    join_attributes="ALL",
    output_type="INPUT"
)

# Removendo Camada de Ligações
for lyr in mapa.listLayers():
    if lyr.name == "Areas_Prioritarias_Count_OS":
        mapa.removeLayer(lyr)
        break

print("✅ Intersect realizado com sucesso!")

In [ ]:
# Criando Ranking

In [ ]:
# Camada
fc = "Areas_Prioritarias_Ranking_RAs"

# Campo de contagem
campo_base = "Join_Count"

# Campo de ranking
campo_ranking = "Ranking"

# Criar campo se não existir
if campo_ranking not in [f.name for f in arcpy.ListFields(fc)]:
    arcpy.management.AddField(fc, campo_ranking, "LONG")

# Ordenar por Join_Count (Decrescente)
dados_ordenados = sorted(
    [(row[0], row[1]) for row in arcpy.da.SearchCursor(fc, ["OID@", campo_base])],
    key=lambda x: x[1],
    reverse=True
)

# Criar dicionário de ranking
ranking_dict = {}
posicao = 1

for oid, valor in dados_ordenados:
    ranking_dict[oid] = posicao
    posicao += 1

# Atualizar tabela
with arcpy.da.UpdateCursor(fc, ["OID@", campo_ranking]) as ucursor:
    for row in ucursor:
        row[1] = ranking_dict.get(row[0])
        ucursor.updateRow(row)

print("✅ Ranking criado com sucesso!")

In [ ]:
# Selecionando 10 piores polígonos para cada RA

In [ ]:
from collections import defaultdict

# Entrada
fc = "Areas_Prioritarias_Ranking_RAs"

# Campos
campo_area = "Shape_Area"
campo_rank = "Ranking"
campo_regiao = "Nome_Python"
oid_field = arcpy.Describe(fc).OIDFieldName

# Parâmetros
area_min = 10000 # Área mínima para que o polígono seja rankeado 
top_n = 10  # Defina o ranking desejado

# Agrupamento por região
grupos = defaultdict(list)

# Filtrar por área e agrupar
with arcpy.da.SearchCursor(fc, [oid_field, campo_regiao, campo_rank, campo_area]) as cursor:
    for oid, regiao, rank, area in cursor:
        if area is not None and area > area_min:
            grupos[regiao].append((oid, rank))

# Selecionar os 10 piores (ranking menor)
oids_selecionados = []

for regiao, lista in grupos.items():
    # Ordenar (menor ranking = pior área)
    lista_ordenada = sorted(lista, key=lambda x: x[1])
    
    # Pegar os 10 primeiros
    top = lista_ordenada[:top_n]
    
    oids_selecionados.extend([oid for oid, rank in top])

print(f"Total selecionado: {len(oids_selecionados)}")

# Criar layer temporária
arcpy.management.MakeFeatureLayer(fc, "lyr_temp")

# Evitar limite de SQL
chunk_size = 3000
chunks = [oids_selecionados[i:i + chunk_size] for i in range(0, len(oids_selecionados), chunk_size)]

sql = " OR ".join([
    f"{oid_field} IN ({','.join(map(str, chunk))})"
    for chunk in chunks
])

# Selecionar
arcpy.management.SelectLayerByAttribute("lyr_temp", "NEW_SELECTION", sql)

# Exportar
saida =  f"{gdb}\\{dataset}\\Top_10_Areas_por_RA"
arcpy.management.CopyFeatures("lyr_temp", saida)

# Removendo Camada de Ligações
for lyr in list(mapa.listLayers()):
    if lyr.name in ["lyr_temp", "Areas_Prioritarias_Ranking_RAs"]:
        mapa.removeLayer(lyr)

print("✅ Seleção das 10 piores áreas por região concluída!")

In [ ]:
# Criando Coluna Ranking por RA

In [ ]:
# Camada
fc = "Top_10_Areas_por_RA"

# Campos
campo_rank_global = "Ranking"
campo_regiao = "Nome_Python"
campo_rank_regiao = "Ranking_Regiao"
oid_field = arcpy.Describe(fc).OIDFieldName

# Criar campo novo
if campo_rank_regiao not in [f.name for f in arcpy.ListFields(fc)]:
    arcpy.management.AddField(fc, campo_rank_regiao, "LONG")

# Agrupar dados por região
grupos = defaultdict(list)

with arcpy.da.SearchCursor(fc, [oid_field, campo_regiao, campo_rank_global]) as cursor:
    for oid, regiao, rank_global in cursor:
        grupos[regiao].append((oid, rank_global))

# Criar dicionário final de ranking por região
ranking_dict = {}

for regiao, lista in grupos.items():
    # Ordenar pelo ranking global (menor = pior)
    lista_ordenada = sorted(lista, key=lambda x: x[1])
    
    # Atribuir ranking local (1 a 10)
    for i, (oid, rank_global) in enumerate(lista_ordenada, start=1):
        ranking_dict[oid] = i

# Atualizar a tabela
with arcpy.da.UpdateCursor(fc, [oid_field, campo_rank_regiao]) as ucursor:
    for row in ucursor:
        oid = row[0]
        row[1] = ranking_dict.get(oid)
        ucursor.updateRow(row)

print("✅ Ranking por região criado com sucesso!")

In [ ]:
# Colocar parte da simbolgia das Áreas Prioritárias

In [ ]:
# Exportando Layouts de Áreas Prioritárias

In [ ]:
'''
# Caminho do projeto .aprx
aprx_path = r"C:\Projetos_ArcGis_Felipe_2025\Multi_Criterio_Ramais\Multi_Criterio_Ramais.aprx"
aprx = arcpy.mp.ArcGISProject(aprx_path)

# Nome do layout modelo
layout_model = aprx.listLayouts("A1_MULTI_CRITERIO")[0]  # ajuste se necessário

# Pasta de saída
out_folder = r"C:\Projetos_ArcGis_Felipe_2025\Multi_Criterio_Ramais\Mapas_IDW"

# Mapa principal (rasters + urbanismo)
mapa_principal = aprx.listMaps("Mapa_Principal")[0]

# Map frame do layout
map_frame = layout_model.listElements("MAPFRAME_ELEMENT", "Map_Frame_Principal")[0]

# Elemento de texto do selo (ajuste o nome conforme definido no layout)
selo_elemento = layout_model.listElements("TEXT_ELEMENT", "Texto_Selo")[0]

# Bookmarks do mapa principal
bookmarks = mapa_principal.listBookmarks()

# Nomes das camadas de urbanismo
camadas_urbanismo = ["Via_por_AOI_v4_Sul_Oeste", "Quadras_Label_Mapas_de_Calor_Sul_Oeste", "Top50_Piores_por_Regiao"]

for bm in bookmarks:
    nome = bm.name

    # Desligar todas as camadas
    for lyr in mapa_principal.listLayers():
        lyr.visible = False

    # Ligar apenas o raster correspondente ao bookmark
    camada = [lyr for lyr in mapa_principal.listLayers() if lyr.name == nome]
    if not camada:
        print(f"⚠ Nenhuma camada encontrada para {nome}")
        continue
    camada[0].visible = True

    selo_texto = nome  # valor padrão caso não ache na tabela

    # Ativar camadas de urbanismo com filtro RA_Python_IDW
    for urb_name in camadas_urbanismo:
        urb_layer = [lyr for lyr in mapa_principal.listLayers() if lyr.name == urb_name]
        if urb_layer:
            urb_layer = urb_layer[0]
            urb_layer.visible = True
            urb_layer.definitionQuery = f"RA_Python_IDW = '{nome}'"

            # Se for a camada Quadra_por_RA, buscar o valor do campo Selo
            if urb_name == "Via_por_AOI_v4_Sul_Oeste":
                with arcpy.da.SearchCursor(urb_layer.dataSource, ["RA_Python_IDW", "Selo"]) as cursor:
                    for row in cursor:
                        if row[0] == nome:
                            selo_texto = row[1]
                            break
        else:
            print(f"⚠ Camada de urbanismo {urb_name} não encontrada no mapa")

    # Atualizar selo no layout
    selo_elemento.text = selo_texto

    # Aplicar o bookmark no map frame
    map_frame.zoomToBookmark(bm)

    # Exportar PNG
    out_png = os.path.join(out_folder, f"{nome}.png")
    layout_model.exportToPNG(out_png, resolution=300)
    print(f"✅ Layout exportado: {out_png} (Selo: {selo_texto})")
'''

* FIM